LLM Analyses merging script
---------------------------
This script uses the classification and readability output files to create one merged .csv file. Instances where a classification output is available, but no readability analysis, are assigned N/A values for the readability part.

In [1]:
import os
import re
import pandas as pd

configuration:

In [ ]:
ROOT = "C:--FILL IN YOUR ROOT FOLDER--"  # root project folder
CLSF_DIR = f"{ROOT}/LLM Analyses/Classification"
RDBL_DIR = f"{ROOT}/LLM Analyses/Readability Analysis"
OUTPUT_DIR = "."

CLSF_FILE = "Classification_GPT5n_role5_examples.csv"
RDBL_FILE = "Readability analysis_GPT5.1_role2.csv"
OUTPUT_FILE = "LinkedIn LLM Analysis output.csv"

main script

In [ ]:
def count_sentences(text):
    """Estimates sentence count based on punctuation (. ! ?)."""
    if not isinstance(text, str) or not text.strip():
        return 0
    sentences = re.findall(r"[^.!?]+[.!?]", text)
    return max(len(sentences), 1)

def main():
    # Construct full file paths
    clsf_path = os.path.join(CLSF_DIR, CLSF_FILE)
    rdbl_path = os.path.join(RDBL_DIR, RDBL_FILE)
    output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)

    print("--- Step 1: Loading Datasets ---")
    df1 = pd.read_csv(clsf_path, encoding="utf-8-sig")  # Thematic classification dataset (Larger)
    df2 = pd.read_csv(rdbl_path, encoding="utf-8-sig")  # Readability dataset (Smaller)

    # Ensure URL is unique in df1 and df2 for a clean one-to-one join
    df1 = df1.drop_duplicates(subset=["URL"])
    print(f"Dataset 1 loaded with {len(df1)} unique rows.")
    df2 = df2.drop_duplicates(subset=["URL"])
    print(f"Dataset 2 loaded with {len(df2)} unique rows.")

    print("\n--- Step 2: Analyzing Columns ---")
    # Define common columns shared between both datasets
    common_cols = ["Company Name", "Year", "Date", "Text", "URL"]

    # Identify extra columns unique to each dataset (excluding common columns and counts)
    extra_cols_df1 = [
        col
        for col in df1.columns
        if col not in common_cols and col != "word_count"
    ]
    extra_cols_df2 = [
        col
        for col in df2.columns
        if col not in common_cols and col != "sentence_count"
    ]

    # Select only the necessary columns from df2 for the merge
    df2_to_join = df2[["URL", "sentence_count"] + extra_cols_df2]

    print("\n--- Step 3: Merging Datasets (Left Join) ---")
    # Perform a left join keeping all records from df1
    merged_df = pd.merge(df1, df2_to_join, on="URL", how="left")

    print("\n--- Step 4: Calculating Missing Sentence Counts ---")
    # Identify rows where sentence_count is missing (NaN) after the join
    missing_sentence_mask = merged_df["sentence_count"].isna()

    # Apply the count_sentences function only to the missing rows using the 'Text' column
    merged_df.loc[missing_sentence_mask, "sentence_count"] = merged_df.loc[
        missing_sentence_mask, "Text"
    ].apply(count_sentences)

    # Convert sentence_count column to integer type
    merged_df["sentence_count"] = merged_df["sentence_count"].astype(int)

    print(
        f"Manual sentence count calculated for {missing_sentence_mask.sum()} rows."
    )

    print("\n--- Step 5: Ordering Columns ---")
    # Reconstruct the final column order as specified
    final_column_order = (
        common_cols
        + ["word_count", "sentence_count"]
        + extra_cols_df1
        + extra_cols_df2
    )

    # Reorder the dataframe to match the final column order
    final_df = merged_df[final_column_order]

    print("\n--- Step 6: Saving Result ---")
    # Create the output directory if it does not exist yet
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    final_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Success! Processed dataset saved to: {output_path}")


if __name__ == "__main__":
    main()